# Access an ASAM ODS EXD-API Plugin

## Prepare Python Environment to Access GRPC Service

In [13]:
import os
import pathlib

from ods_exd_api_box.proto import ods_external_data_pb2 as exd_api, ods_external_data_pb2_grpc
import values_append_pb2 as va
import values_append_pb2_grpc

import mdm_pb2 as Mdm


import grpc
from google.protobuf.json_format import MessageToJson

## EXD-API

The EXD-API plugin is running as a RPC service at a given URL.
Running `external_data_file.py` will run the plugin at the given URL.

In [2]:
exd_api_plugin_url = "localhost:50051"

In [ ]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = values_append_pb2_grpc.ValuesAppenderStub(channel)

    group_ids = va.InitRequest(group_ids=[42])
    duckdb = stub.Init(group_ids)
    with open("data/testfile.duckdb", "wb") as f:
        f.write(duckdb.duckdb_file_content)


## Import Phase

We will open a DuckDB file using the EXD-API and extract the internal structure of the file to import it into the ASAM ODS server.

In [5]:
data_file_path = os.path.abspath("data/testfile.duckdb")
if not os.path.exists(data_file_path):
    raise Exception("Data file is missing")

In [6]:
import_file_url = pathlib.Path(data_file_path).as_uri()
import_file_parameters = "mode=normalized"
print(import_file_url)

# Will be filled from Structure
access_file_url = None
access_file_parameters = None

file:///workspaces/duckdb/asam_ods_exd_api_duckdb/data/testfile.duckdb


### Extract Infos from Structure

The structure contains infos about groups and channels to create corresponding measurements, submatrices and measurement_quantities

In [ ]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # import file into ASAM ODS Server physical storage
    import_identifier = exd_api.Identifier(url=import_file_url, parameters=import_file_parameters)

    import_handle = stub.Open(import_identifier)
    try:
        structure = stub.GetStructure(exd_api.StructureRequest(handle=import_handle))
        print(MessageToJson(structure))

        access_file_url = structure.identifier.url
        access_file_parameters = structure.identifier.parameters

        for group in structure.groups:
            group_id = group.id
            for channel in group.channels:
                channel_id = channel.id
    finally:
        stub.Close(import_handle)

{
  "identifier": {
    "url": "file:///workspaces/duckdb/asam_ods_exd_api_duckdb/data/testfile.duckdb",
    "parameters": "mode=normalized"
  },
  "name": "testfile.duckdb",
  "groups": [
    {
      "id": "42",
      "name": "sm42",
      "totalNumberOfChannels": "2",
      "numberOfRows": "10",
      "channels": [
        {
          "id": "1",
          "name": "1",
          "dataType": "DT_LONGLONG"
        },
        {
          "id": "2",
          "name": "2",
          "dataType": "DT_FLOAT"
        }
      ]
    }
  ],
  "attributes": {
    "variables": {
      "start_time": {
        "stringArray": {
          "values": [
            "20260716164826448288"
          ]
        }
      }
    }
  }
}


## Append Bulk data

Use Append method to append bulk data

In [50]:
write_requests = Mdm.WriteRequestList()
req1 = write_requests.values.add(channel_group_id = "42", channel_id = "1")
req1.explicit.long_array.values.extend([123, 2])
req1.explicit.flags_full.extend([15, 15])

req2 = write_requests.values.add(channel_group_id = "42", channel_id = "2")
req2.explicit.float_array.values.extend([81.2, 1.2])
req2.explicit.flags_full.extend([15, 0])

with grpc.insecure_channel(exd_api_plugin_url) as channel:
    exd_stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)
    va_stub = values_append_pb2_grpc.ValuesAppenderStub(channel)

    # open bulk access
    access_identifier = exd_api.Identifier(url=access_file_url, parameters=access_file_parameters)
    access_handle = exd_stub.Open(access_identifier)

    try:
        append_response = va_stub.Append(va.AppendRequest(handle = access_handle, values= write_requests))
        print(append_response)

    finally:
        exd_stub.Close(access_handle)

rowCount: 10



## Access Bulk Data

With the stored information the ASAM ODS server can access the bulk data from the EXD-API plugin

In [51]:
with grpc.insecure_channel(exd_api_plugin_url) as channel:
    exd_stub = ods_external_data_pb2_grpc.ExternalDataReaderStub(channel)

    # open bulk access
    access_identifier = exd_api.Identifier(url=access_file_url, parameters=access_file_parameters)
    access_handle = exd_stub.Open(access_identifier)

    try:
        values = exd_stub.GetValues(exd_api.ValuesRequest(handle=access_handle, group_id=42, channel_ids=[1, 2], start=0, limit=100))
        print(MessageToJson(values))

    finally:
        exd_stub.Close(access_handle)

{
  "id": "42",
  "channels": [
    {
      "id": "1",
      "values": {
        "dataType": "DT_LONGLONG",
        "longlongArray": {
          "values": [
            "123",
            "2",
            "123",
            "2",
            "123",
            "2",
            "123",
            "2",
            "123",
            "2"
          ]
        }
      }
    },
    {
      "id": "2",
      "values": {
        "dataType": "DT_FLOAT",
        "floatArray": {
          "values": [
            81.2,
            1.2,
            81.2,
            1.2,
            81.2,
            1.2,
            81.2,
            1.2,
            81.2,
            1.2
          ]
        }
      },
      "flags": {
        "values": [
          15,
          0,
          15,
          0,
          15,
          0,
          15,
          0,
          15,
          0
        ]
      }
    }
  ]
}
